In [23]:
!pip -q install yfinance psycopg2-binary sqlalchemy

In [24]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
import psycopg2

from psycopg2.extras import execute_values

In [25]:
ticker = "RELIANCE.NS"

print("Stock:", ticker)

Stock: RELIANCE.NS


In [26]:
ticker = "RELIANCE.NS"

print("Stock:", ticker)

Stock: RELIANCE.NS


In [27]:
data = yf.download(
    ticker,
    period="5d",
    interval="5m",
    auto_adjust=False
)

print("Data downloaded successfully!")
print("Rows:", len(data))

[*********************100%***********************]  1 of 1 completed

Data downloaded successfully!
Rows: 364


In [28]:
print(data.head())

print("\nColumns:")
print(data.columns)

print("\nData types:")
print(data.dtypes)

Price                        Adj Close        Close         High          Low  \
Ticker                     RELIANCE.NS  RELIANCE.NS  RELIANCE.NS  RELIANCE.NS   
Datetime                                                                        
2026-08-28 03:45:00+00:00  1286.400024  1286.400024  1291.500000  1285.400024   
2026-08-28 03:50:00+00:00  1287.800049  1287.800049  1288.800049  1286.599976   
2026-08-28 03:55:00+00:00  1287.099976  1287.099976  1289.500000  1287.000000   
2026-08-28 04:00:00+00:00  1286.900024  1286.900024  1287.800049  1286.300049   
2026-08-28 04:05:00+00:00  1287.099976  1287.099976  1287.300049  1285.500000   

Price                             Open      Volume  
Ticker                     RELIANCE.NS RELIANCE.NS  
Datetime                                            
2026-08-28 03:45:00+00:00  1285.400024           0  
2026-08-28 03:50:00+00:00  1286.599976      157668  
2026-08-28 03:55:00+00:00  1287.900024      128580  
2026-08-28 04:00:00+00:00  1287.1

In [29]:
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data.reset_index()

data.columns = [
    str(col).lower().replace(" ", "_")
    for col in data.columns
]

data["symbol"] = ticker

print(data.head())
print("\nColumns:")
print(data.columns.tolist())

                   datetime    adj_close        close         high  \
0 2026-08-28 03:45:00+00:00  1286.400024  1286.400024  1291.500000   
1 2026-08-28 03:50:00+00:00  1287.800049  1287.800049  1288.800049   
2 2026-08-28 03:55:00+00:00  1287.099976  1287.099976  1289.500000   
3 2026-08-28 04:00:00+00:00  1286.900024  1286.900024  1287.800049   
4 2026-08-28 04:05:00+00:00  1287.099976  1287.099976  1287.300049   

           low         open  volume       symbol  
0  1285.400024  1285.400024       0  RELIANCE.NS  
1  1286.599976  1286.599976  157668  RELIANCE.NS  
2  1287.000000  1287.900024  128580  RELIANCE.NS  
3  1286.300049  1287.199951  145918  RELIANCE.NS  
4  1285.500000  1286.900024   76972  RELIANCE.NS  

Columns:
['datetime', 'adj_close', 'close', 'high', 'low', 'open', 'volume', 'symbol']


In [30]:
data["datetime"] = data["datetime"].dt.tz_convert("Asia/Kolkata")

print(data["datetime"].head())

0   2026-08-28 09:15:00+05:30
1   2026-08-28 09:20:00+05:30
2   2026-08-28 09:25:00+05:30
3   2026-08-28 09:30:00+05:30
4   2026-08-28 09:35:00+05:30
Name: datetime, dtype: datetime64[ns, Asia/Kolkata]


In [31]:
print("First timestamp:",
      data["datetime"].dt.time.min())

print("Last timestamp:",
      data["datetime"].dt.time.max())

First timestamp: 09:15:00
Last timestamp: 15:15:00


In [32]:
print("Missing values:")
print(data.isnull().sum())

Missing values:
datetime     0
adj_close    0
close        0
high         0
low          0
open         0
volume       0
symbol       0
dtype: int64


In [33]:
duplicate_count = data["datetime"].duplicated().sum()

print("Duplicate timestamps:", duplicate_count)

Duplicate timestamps: 0


In [34]:
data["date"] = data["datetime"].dt.date

data["time_diff"] = (
    data.groupby("date")["datetime"].diff()
)

missing_intervals = data[
    data["time_diff"] > pd.Timedelta(minutes=5)
]

print("Missing intervals detected:",
      len(missing_intervals))

if len(missing_intervals) > 0:
    print(
        missing_intervals[
            ["datetime", "time_diff"]
        ]
    )
else:
    print("No obvious intraday gaps detected.")

Missing intervals detected: 0
No obvious intraday gaps detected.


In [35]:
invalid_ohlc = data[
    (data["high"] < data["open"]) |
    (data["high"] < data["close"]) |
    (data["low"] > data["open"]) |
    (data["low"] > data["close"]) |
    (data["high"] < data["low"])
]

print("Invalid OHLC rows:",
      len(invalid_ohlc))

Invalid OHLC rows: 0


In [36]:
negative_volume = (
    data["volume"] < 0
).sum()

zero_volume = (
    data["volume"] == 0
).sum()

print("Negative volume:", negative_volume)
print("Zero volume:", zero_volume)

Negative volume: 0
Zero volume: 5


In [37]:
data["zero_volume_flag"] = (
    data["volume"] == 0
)

print(
    data["zero_volume_flag"]
    .value_counts()
)

zero_volume_flag
False    359
True       5
Name: count, dtype: int64


In [38]:
data = data.drop(
    columns=["date", "time_diff"]
)

print(data.columns.tolist())

['datetime', 'adj_close', 'close', 'high', 'low', 'open', 'volume', 'symbol', 'zero_volume_flag']


In [16]:
conn = psycopg2.connect(
    host="YOUR_HOST",
    port="YOUR_PORT",
    database="tsdb",
    user="tsdbadmin",
    password="YOUR_PASSWORD",
    sslmode="require"
)
cursor = conn.cursor()

print("Connected successfully!")

In [41]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS market_data (
    datetime TIMESTAMPTZ NOT NULL,
    symbol TEXT NOT NULL,
    open DOUBLE PRECISION,
    high DOUBLE PRECISION,
    low DOUBLE PRECISION,
    close DOUBLE PRECISION,
    volume BIGINT,
    zero_volume_flag BOOLEAN
);
""")

conn.commit()

print("Table created successfully!")

Table created successfully!


In [42]:
cursor.execute("""
SELECT create_hypertable(
    'market_data',
    by_range('datetime'),
    if_not_exists => TRUE
);
""")

conn.commit()

print("market_data is now a TimescaleDB hypertable!")

market_data is now a TimescaleDB hypertable!


In [43]:
cursor.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
market_data_symbol_datetime_idx
ON market_data (symbol, datetime);
""")

conn.commit()

print("Unique index created!")

Unique index created!


In [44]:
db_data = data[
    [
        "datetime",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "zero_volume_flag"
    ]
].copy()

records = list(
    db_data.itertuples(
        index=False,
        name=None
    )
)

print("Records prepared:", len(records))

Records prepared: 364


In [45]:
execute_values(
    cursor,
    """
    INSERT INTO market_data
    (
        datetime,
        symbol,
        open,
        high,
        low,
        close,
        volume,
        zero_volume_flag
    )
    VALUES %s

    ON CONFLICT (symbol, datetime)
    DO NOTHING;
    """,
    records
)

conn.commit()

print(
    f"Inserted/processed {len(records)} records."
)

Inserted/processed 364 records.


In [46]:
cursor.execute("""
SELECT COUNT(*)
FROM market_data;
""")

row_count = cursor.fetchone()[0]

print("Rows in database:", row_count)

Rows in database: 364


In [47]:
cursor.execute("""
SELECT
    datetime,
    symbol,
    open,
    high,
    low,
    close,
    volume
FROM market_data
ORDER BY datetime DESC
LIMIT 10;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(datetime.datetime(2026, 9, 3, 9, 45, tzinfo=datetime.timezone.utc), 'RELIANCE.NS', 1309.0999755859375, 1309.0999755859375, 1302.5, 1302.5, 690251)
(datetime.datetime(2026, 9, 3, 9, 40, tzinfo=datetime.timezone.utc), 'RELIANCE.NS', 1310.300048828125, 1311.699951171875, 1308.699951171875, 1309.0, 435569)
(datetime.datetime(2026, 9, 3, 9, 35, tzinfo=datetime.timezone.utc), 'RELIANCE.NS', 1310.5, 1311.0999755859375, 1309.0, 1310.4000244140625, 194645)
(datetime.datetime(2026, 9, 3, 9, 30, tzinfo=datetime.timezone.utc), 'RELIANCE.NS', 1311.0, 1311.800048828125, 1310.0999755859375, 1310.199951171875, 131967)
(datetime.datetime(2026, 9, 3, 9, 25, tzinfo=datetime.timezone.utc), 'RELIANCE.NS', 1308.4000244140625, 1312.0, 1308.4000244140625, 1311.5, 304415)
(datetime.datetime(2026, 9, 3, 9, 20, tzinfo=datetime.timezone.utc), 'RELIANCE.NS', 1308.5999755859375, 1309.0, 1308.4000244140625, 1308.800048828125, 108468)
(datetime.datetime(2026, 9, 3, 9, 15, tzinfo=datetime.timezone.utc), 'RELIANCE.NS'

In [48]:
cursor.execute("""
SELECT
    hypertable_schema,
    hypertable_name,
    num_dimensions,
    primary_dimension,
    primary_dimension_type
FROM timescaledb_information.hypertables
WHERE hypertable_name = 'market_data';
""")

for row in cursor.fetchall():
    print(row)

('public', 'market_data', 1, 'datetime', 'timestamp with time zone')


In [49]:
def fetch_market_data(
    ticker,
    retries=3,
    force_fail=False
):

    for attempt in range(1, retries + 1):

        try:

            print(
                f"Attempt {attempt}: "
                f"Fetching {ticker}"
            )

            # Simulate an API failure
            if force_fail:
                raise TimeoutError(
                    "Simulated API timeout"
                )

            df = yf.download(
                ticker,
                period="1d",
                interval="5m",
                auto_adjust=False
            )

            if df.empty:
                raise ValueError(
                    "API returned empty data"
                )

            print("Data fetched successfully!")

            return df

        except Exception as e:

            print(
                f"Attempt {attempt} failed: {e}"
            )

            if attempt < retries:

                wait_time = 2 ** (attempt - 1)

                print(
                    f"Retrying in "
                    f"{wait_time} seconds..."
                )

                time.sleep(wait_time)

    print("All attempts failed.")

    return None

In [50]:
result = fetch_market_data(
    ticker,
    force_fail=True
)

print("\nResult:", result)

Attempt 1: Fetching RELIANCE.NS
Attempt 1 failed: Simulated API timeout
Retrying in 1 seconds...
Attempt 2: Fetching RELIANCE.NS
Attempt 2 failed: Simulated API timeout
Retrying in 2 seconds...
Attempt 3: Fetching RELIANCE.NS
Attempt 3 failed: Simulated API timeout
All attempts failed.

Result: None


In [51]:
backup_data = data.copy()

In [52]:
def fetch_with_failover(
    ticker,
    retries=3,
    force_primary_fail=False
):

    # -------------------------
    # PRIMARY SOURCE
    # -------------------------

    for attempt in range(1, retries + 1):

        try:

            print(
                f"Primary API - Attempt {attempt}"
            )

            if force_primary_fail:
                raise TimeoutError(
                    "Simulated primary API failure"
                )

            df = yf.download(
                ticker,
                period="1d",
                interval="5m",
                auto_adjust=False
            )

            if df.empty:
                raise ValueError(
                    "Primary API returned no data"
                )

            print("Primary API succeeded!")

            return df, "primary"

        except Exception as e:

            print(
                f"Primary API failed: {e}"
            )

            if attempt < retries:

                wait_time = 2 ** (attempt - 1)

                print(
                    f"Retrying in "
                    f"{wait_time} seconds..."
                )

                time.sleep(wait_time)

    # -------------------------
    # BACKUP SOURCE
    # -------------------------

    print(
        "\nPrimary API unavailable."
    )

    print(
        "Switching to backup source..."
    )

    if backup_data.empty:

        print(
            "Backup source unavailable."
        )

        return None, None

    print(
        "Backup source succeeded!"
    )

    return backup_data.copy(), "backup"

In [53]:
result, source = fetch_with_failover(
    ticker,
    force_primary_fail=True
)

print("\nFinal source:", source)

if result is not None:
    print(
        "Rows received:",
        len(result)
    )

Primary API - Attempt 1
Primary API failed: Simulated primary API failure
Retrying in 1 seconds...
Primary API - Attempt 2
Primary API failed: Simulated primary API failure
Retrying in 2 seconds...
Primary API - Attempt 3
Primary API failed: Simulated primary API failure

Primary API unavailable.
Switching to backup source...
Backup source succeeded!

Final source: backup
Rows received: 364


In [54]:
cursor.execute("""
SELECT
    symbol,

    time_bucket(
        '30 minutes',
        datetime
    ) AS time_interval,

    MIN(low) AS lowest_price,

    MAX(high) AS highest_price,

    SUM(volume) AS total_volume

FROM market_data

GROUP BY
    symbol,
    time_interval

ORDER BY
    time_interval;
""")

rows = cursor.fetchall()

for row in rows[:10]:
    print(row)

('RELIANCE.NS', datetime.datetime(2026, 8, 28, 3, 30, tzinfo=datetime.timezone.utc), 1285.4000244140625, 1291.5, Decimal('286248'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 4, 0, tzinfo=datetime.timezone.utc), 1285.4000244140625, 1288.0999755859375, Decimal('427271'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 4, 30, tzinfo=datetime.timezone.utc), 1285.0, 1290.800048828125, Decimal('439773'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 5, 0, tzinfo=datetime.timezone.utc), 1286.0, 1289.199951171875, Decimal('269026'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 5, 30, tzinfo=datetime.timezone.utc), 1284.0999755859375, 1289.800048828125, Decimal('719435'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 6, 0, tzinfo=datetime.timezone.utc), 1281.199951171875, 1285.0, Decimal('415417'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 6, 30, tzinfo=datetime.timezone.utc), 1281.0999755859375, 1282.9000244140625, Decimal('768385'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 

In [55]:
cursor.execute("""
SELECT
    symbol,

    time_bucket(
        '30 minutes',
        datetime
    ) AS time_interval,

    SUM(close * volume) /
        NULLIF(SUM(volume), 0)
        AS vwap,

    SUM(volume) AS total_volume

FROM market_data

GROUP BY
    symbol,
    time_interval

ORDER BY
    time_interval;
""")

rows = cursor.fetchall()

for row in rows[:10]:
    print(row)

('RELIANCE.NS', datetime.datetime(2026, 8, 28, 3, 30, tzinfo=datetime.timezone.utc), 1287.4855822904358, Decimal('286248'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 4, 0, tzinfo=datetime.timezone.utc), 1286.8408822574604, Decimal('427271'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 4, 30, tzinfo=datetime.timezone.utc), 1287.6838407285593, Decimal('439773'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 5, 0, tzinfo=datetime.timezone.utc), 1287.4386106154195, Decimal('269026'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 5, 30, tzinfo=datetime.timezone.utc), 1286.8209753431458, Decimal('719435'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 6, 0, tzinfo=datetime.timezone.utc), 1282.9576347639654, Decimal('415417'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 6, 30, tzinfo=datetime.timezone.utc), 1281.5043994734149, Decimal('768385'))
('RELIANCE.NS', datetime.datetime(2026, 8, 28, 7, 0, tzinfo=datetime.timezone.utc), 1283.3792405280149, Decimal('694273'))
('RELIANCE.N

In [56]:
cursor.execute("""
WITH daily_prices AS (
    SELECT
        symbol,
        DATE(datetime) AS trading_day,
        MAX(close) AS closing_price
    FROM market_data
    GROUP BY symbol, DATE(datetime)
)

SELECT
    symbol,
    trading_day,
    closing_price,
    closing_price /
        NULLIF(
            LAG(closing_price)
            OVER (
                PARTITION BY symbol
                ORDER BY trading_day
            ),
            0
        ) - 1 AS daily_return
FROM daily_prices
ORDER BY trading_day;
""")

rows = cursor.fetchall()

daily_returns = pd.DataFrame(
    rows,
    columns=[
        "symbol",
        "trading_day",
        "closing_price",
        "daily_return"
    ]
)

print("Daily returns calculated:")
display(daily_returns)

Daily returns calculated:


,symbol,trading_day,closing_price,daily_return
0,RELIANCE.NS,2026-08-28,1288.800049,NaN
1,RELIANCE.NS,2026-08-31,1297.000000,0.006362
2,RELIANCE.NS,2026-09-01,1311.599976,0.011257
3,RELIANCE.NS,2026-09-02,1318.199951,0.005032
4,RELIANCE.NS,2026-09-03,1315.900024,-0.001745


In [57]:
volatility = (
    daily_returns
    .groupby("symbol")["daily_return"]
    .std()
)

print("Daily return volatility:")
print(volatility)

Daily return volatility:
symbol
RELIANCE.NS    0.005363
Name: daily_return, dtype: float64


In [58]:
daily_returns["rolling_volatility"] = (
    daily_returns
    .groupby("symbol")["daily_return"]
    .rolling(window=3)
    .std()
    .reset_index(level=0, drop=True)
)

print("Rolling volatility:")
display(
    daily_returns[
        [
            "symbol",
            "trading_day",
            "daily_return",
            "rolling_volatility"
        ]
    ]
)

Rolling volatility:


,symbol,trading_day,daily_return,rolling_volatility
0,RELIANCE.NS,2026-08-28,NaN,NaN
1,RELIANCE.NS,2026-08-31,0.006362,NaN
2,RELIANCE.NS,2026-09-01,0.011257,NaN
3,RELIANCE.NS,2026-09-02,0.005032,0.003278
4,RELIANCE.NS,2026-09-03,-0.001745,0.006503


In [59]:
daily_returns["return_mean"] = (
    daily_returns
    .groupby("symbol")["daily_return"]
    .transform("mean")
)

daily_returns["return_std"] = (
    daily_returns
    .groupby("symbol")["daily_return"]
    .transform("std")
)

daily_returns["return_zscore"] = (
    (daily_returns["daily_return"] -
     daily_returns["return_mean"])
    / daily_returns["return_std"]
)

daily_returns["anomaly_flag"] = (
    daily_returns["return_zscore"].abs() > 2
)

display(
    daily_returns[
        [
            "symbol",
            "trading_day",
            "daily_return",
            "return_zscore",
            "anomaly_flag"
        ]
    ]
)

,symbol,trading_day,daily_return,return_zscore,anomaly_flag
0,RELIANCE.NS,2026-08-28,NaN,NaN,False
1,RELIANCE.NS,2026-08-31,0.006362,0.211789,False
2,RELIANCE.NS,2026-09-01,0.011257,1.124360,False
3,RELIANCE.NS,2026-09-02,0.005032,-0.036286,False
4,RELIANCE.NS,2026-09-03,-0.001745,-1.299863,False


In [60]:
anomalies = daily_returns[
    daily_returns["anomaly_flag"] == True
]

print("Potential anomalies detected:", len(anomalies))

display(
    anomalies[
        [
            "symbol",
            "trading_day",
            "daily_return",
            "return_zscore"
        ]
    ]
)

Potential anomalies detected: 0


,symbol,trading_day,daily_return,return_zscore


In [61]:
cursor.execute("""
SELECT extversion
FROM pg_extension
WHERE extname = 'timescaledb';
""")

version = cursor.fetchone()[0]

print("TimescaleDB version:", version)

TimescaleDB version: 2.29.2


In [62]:
cursor.execute("""
SELECT
    COUNT(*) AS total_records,

    COUNT(*) FILTER (
        WHERE open IS NULL
        OR high IS NULL
        OR low IS NULL
        OR close IS NULL
        OR volume IS NULL
    ) AS missing_records,

    COUNT(*) FILTER (
        WHERE high < low
    ) AS invalid_price_records,

    COUNT(*) FILTER (
        WHERE volume < 0
    ) AS negative_volume_records

FROM market_data;
""")

result = cursor.fetchone()

print("Total records:", result[0])
print("Records with missing values:", result[1])
print("Invalid price records:", result[2])
print("Negative volume records:", result[3])

Total records: 364
Records with missing values: 0
Invalid price records: 0
Negative volume records: 0


In [63]:
cursor.execute("""
SELECT
    symbol,
    COUNT(*) AS records,
    MIN(datetime) AS first_record,
    MAX(datetime) AS last_record,
    MIN(low) AS minimum_price,
    MAX(high) AS maximum_price,
    SUM(volume) AS total_volume

FROM market_data

GROUP BY symbol;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('RELIANCE.NS', 364, datetime.datetime(2026, 8, 28, 3, 45, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 9, 3, 9, 45, tzinfo=datetime.timezone.utc), 1271.0999755859375, 1321.699951171875, Decimal('74631795'))


In [64]:
cursor.execute("""
ALTER TABLE market_data
SET (
    timescaledb.enable_columnstore = true,
    timescaledb.orderby = 'datetime DESC',
    timescaledb.segmentby = 'symbol'
);
""")

conn.commit()

print("Columnstore enabled successfully!")

Columnstore enabled successfully!


In [65]:
cursor.execute("""
CALL add_columnstore_policy(
    'market_data',
    after => INTERVAL '1 day'
);
""")

conn.commit()

print("Columnstore policy added successfully!")

Columnstore policy added successfully!


In [66]:
cursor.close()
conn.close()

print("Database connection closed.")

Database connection closed.
